# Fine-tuning YOLOv8s on the WTBD blade-defect dataset

This Colab notebook fine-tunes a COCO-pretrained **YOLOv8s** detector on the
Wind Turbine Blade Defect (WTBD) dataset that was prepared with
`src/prepare_data.py`.

**What it does, end to end:**
1. Check the GPU.
2. Install Ultralytics.
3. Mount Google Drive and locate the prepared dataset.
4. Load COCO-pretrained `yolov8s.pt`.
5. Run validation **before** training to record a baseline.
6. Fine-tune for ~50 epochs with early stopping and blade-appropriate augmentation.
7. Save the best weights to Drive.
8. Print and save per-class precision, recall, mAP@0.5 and mAP@0.5:0.95.
9. Evaluate on the held-out **TEST** split: per-class recall, AUPRC, confusion matrix, examples.

> **Runtime:** set **Runtime → Change runtime type → T4 GPU** before running.
> Image size and batch are kept modest so everything fits in a free T4 (16 GB).

## 1. Check the GPU

`nvidia-smi` prints the allocated GPU, its memory, and the driver/CUDA version.
On a free Colab you normally get a **Tesla T4 (16 GB)**. If this errors or shows
no GPU, switch the runtime type to a GPU and re-run.

In [ ]:
!nvidia-smi

## 2. Install Ultralytics

Ultralytics ships YOLOv8 and pulls in a compatible PyTorch build. We install it
quietly, then print the versions and confirm PyTorch sees the GPU.

In [ ]:
%pip install -q ultralytics

In [ ]:
import torch
import ultralytics
from ultralytics import YOLO

ultralytics.checks()
print('Ultralytics:', ultralytics.__version__)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > T4 GPU'
print('Device:', torch.cuda.get_device_name(0))

## 3. Mount Google Drive and locate the dataset

Upload the **output** of `src/prepare_data.py` to your Drive. That folder
(call it `wtbd_yolo`) must contain the YOLO-format dataset and its `data.yaml`:

```
MyDrive/wtbd_yolo/
├── images/{train,val,test}/   # the .jpg files
├── labels/{train,val,test}/   # the YOLO .txt labels
└── data.yaml                  # written by prepare_data.py
```

The `data.yaml` produced locally has a Windows `path:` line. Colab can't use
that, so the next cell rewrites `path:` to point at the Drive copy. Ultralytics
resolves the `train`/`val`/`test` entries relative to `path:`, so this is the
only line that needs changing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# dataset location on Drive
DATASET_DIR = Path('/content/drive/MyDrive/wtbd_yolo')

assert DATASET_DIR.exists(), f'Dataset folder not found: {DATASET_DIR}'
for sub in ['images/train', 'images/val', 'labels/train', 'labels/val', 'data.yaml']:
    assert (DATASET_DIR / sub).exists(), f'Missing {sub} under {DATASET_DIR}'
print('Dataset found at:', DATASET_DIR)

# Rewrite the data.yaml `path:` to the Drive location and save a Colab copy.
src_yaml = (DATASET_DIR / 'data.yaml').read_text(encoding='utf-8').splitlines()
patched = []
for line in src_yaml:
    if line.strip().startswith('path:'):
        patched.append(f'path: {DATASET_DIR.as_posix()}')
    else:
        patched.append(line)
DATA_YAML = '/content/wtbd_colab.yaml'
Path(DATA_YAML).write_text('\n'.join(patched) + '\n', encoding='utf-8')
print('\nUsing data.yaml:\n')
print(Path(DATA_YAML).read_text(encoding='utf-8'))

## 4. Load the COCO-pretrained model

`yolov8s.pt` is the **small** YOLOv8 checkpoint pretrained on COCO (80 everyday
object classes). "Small" is a good size/speed trade-off for a T4. We fine-tune
from these weights rather than training from scratch — the pretrained backbone
already knows generic edges, textures and shapes, so it adapts to blade defects
with far less data and time.

In [ ]:
model = YOLO('yolov8s.pt')  # COCO-pretrained checkpoint
print('Loaded yolov8s.pt with', len(model.names), 'COCO classes')

## 5. Baseline validation BEFORE fine-tuning

Before touching the weights we evaluate the pretrained model on the blade
validation set to record a **baseline**.

Be clear about what this number means: COCO has no blade-defect classes, so the
pretrained model literally cannot predict `craze`, `corrosion`, etc. The
baseline mAP will therefore be **≈ 0**. That is the point — it establishes the
floor, so the post-training metrics in Step 8 show exactly how much the
fine-tuning bought us, with no ambiguity about a 'lucky' starting point.

In [ ]:
import json

RESULTS_DIR = DATASET_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

baseline = model.val(data=DATA_YAML, split='val', imgsz=640, verbose=False)
b = baseline.box
baseline_metrics = {
    'precision': float(b.mp),
    'recall': float(b.mr),
    'mAP50': float(b.map50),
    'mAP50-95': float(b.map),
}
print('Baseline (pretrained, no fine-tuning):')
for k, v in baseline_metrics.items():
    print(f'  {k:>10}: {v:.4f}')

(RESULTS_DIR / 'baseline_metrics.json').write_text(json.dumps(baseline_metrics, indent=2))
print('\nSaved ->', RESULTS_DIR / 'baseline_metrics.json')

## 6. Fine-tune with blade-appropriate augmentation

We train for **50 epochs** with **early stopping** (`patience=15`): if the
validation metric does not improve for 15 epochs, training stops and the best
checkpoint is kept. So 50 is an upper bound, not a fixed cost.

`imgsz=640` and `batch=16` keep memory modest for a T4. **If you hit a CUDA
out-of-memory error, lower `batch` to 8** (or set `batch=-1` to auto-pick).

### Why each augmentation, for blade imagery

Turbine blades are inspected outdoors by drones/cameras from many angles and in
changing light. The augmentations below simulate that variability so the model
generalises instead of memorising the training shots.

| Setting | Value | Why it suits blade defects |
|---|---|---|
| `fliplr` (horizontal flip) | 0.5 | A crack or erosion patch looks the same mirrored — free doubling of orientations. |
| `flipud` (vertical flip) | 0.5 | Blades are shot from drones at any roll angle; a defect has no canonical 'up'. Safe to flip vertically here (unlike natural photos). |
| `degrees` (rotation) | 10 | The camera is rarely perfectly aligned with the blade axis; small rotations mimic that. Kept small so boxes stay tight. |
| `scale` (zoom jitter) | 0.5 | Defects appear tiny or large depending on stand-off distance/zoom; scale jitter teaches scale-invariance. |
| `translate` (shift) | 0.1 | A defect can sit anywhere in frame; positional jitter prevents location bias. |
| `hsv_v` (brightness) | 0.4 | Sun, cloud and time-of-day swing exposure a lot outdoors — the main lighting nuisance. |
| `hsv_s` (saturation) | 0.7 | Stands in for contrast/colour variation across cameras and weather. |
| `hsv_h` (hue) | 0.015 | Blade surfaces are white/grey, so only a tiny white-balance shift is realistic. |
| `mosaic` | 1.0 (ON) | Stitches 4 images into one — more context and scale variety per step, a strong regulariser that helps small defects. |
| `close_mosaic` | 10 | Turns mosaic **off** for the final 10 epochs so the model finishes on real, undistorted images for cleaner convergence. |
| `mixup` | 0.0 (OFF) | Blending two blade photos creates unrealistic ghosting that never occurs in real inspection — left off. |

In [ ]:
results = model.train(
    data=DATA_YAML,
    epochs=50,
    patience=15,            # early stopping
    imgsz=640,
    batch=16,
    device=0,
    seed=42,
    project='/content/runs',
    name='yolov8s_wtbd',
    pretrained=True,
    # --- augmentation ---
    fliplr=0.5,
    flipud=0.5,
    degrees=10.0,
    scale=0.5,
    translate=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    mosaic=1.0,
    close_mosaic=10,
    mixup=0.0,
)
print('Training finished. Run dir:', results.save_dir)

## 7. Save the best weights to Drive

Training wrote everything to `/content/runs` (local and fast). Ultralytics keeps
two checkpoints: `best.pt` (highest validation mAP) and `last.pt` (final epoch).
We copy **`best.pt`** — plus the training plots/CSV — to Drive so they survive
after the Colab session ends.

In [ ]:
import shutil

run_dir = Path(results.save_dir)
best_pt = run_dir / 'weights' / 'best.pt'
assert best_pt.exists(), f'best.pt not found in {run_dir}'

WEIGHTS_DIR = DATASET_DIR / 'weights'
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
dst = WEIGHTS_DIR / 'yolov8s_wtbd_best.pt'
shutil.copy2(best_pt, dst)
print('Saved best weights ->', dst)

# Also copy the training curves / results.csv for the record.
for fname in ['results.csv', 'results.png', 'confusion_matrix.png', 'args.yaml']:
    src = run_dir / fname
    if src.exists():
        shutil.copy2(src, RESULTS_DIR / fname)
        print('Saved ->', RESULTS_DIR / fname)

## 8. Per-class metrics after fine-tuning

Finally we validate the **best** model and report, for every defect class:

- **Precision** — of the boxes the model called this defect, how many were right.
- **Recall** — of the real defects of this class, how many it found.
- **mAP@0.5** — average precision at a loose (IoU ≥ 0.5) box-overlap threshold.
- **mAP@0.5:0.95** — the stricter COCO-style average over IoU 0.5→0.95; the
  headline detection metric.

The `all` row is the dataset-wide average. We print the table and save it as
both CSV and JSON to the Drive `results/` folder, alongside the baseline from
Step 5 for easy before/after comparison.

In [ ]:
import pandas as pd

# Reload the best weights.
best_model = YOLO(str(dst))
metrics = best_model.val(data=DATA_YAML, split='val', imgsz=640, verbose=False)
box = metrics.box
names = best_model.names

rows = []
for i, c in enumerate(box.ap_class_index):
    rows.append({
        'class': names[int(c)],
        'precision': float(box.p[i]),
        'recall': float(box.r[i]),
        'mAP50': float(box.ap50[i]),
        'mAP50-95': float(box.ap[i]),
    })
# Dataset-wide row.
rows.append({
    'class': 'all',
    'precision': float(box.mp),
    'recall': float(box.mr),
    'mAP50': float(box.map50),
    'mAP50-95': float(box.map),
})
df = pd.DataFrame(rows)
pd.set_option('display.float_format', lambda v: f'{v:.4f}')
print(df.to_string(index=False))

df.to_csv(RESULTS_DIR / 'per_class_metrics.csv', index=False)
df.to_json(RESULTS_DIR / 'per_class_metrics.json', orient='records', indent=2)
print('\nSaved ->', RESULTS_DIR / 'per_class_metrics.csv')
print('Saved ->', RESULTS_DIR / 'per_class_metrics.json')

In [ ]:
# Before/after at a glance.
print('              baseline   fine-tuned')
print(f"mAP50      :   {baseline_metrics['mAP50']:.4f}     {float(box.map50):.4f}")
print(f"mAP50-95   :   {baseline_metrics['mAP50-95']:.4f}     {float(box.map):.4f}")
print(f"precision  :   {baseline_metrics['precision']:.4f}     {float(box.mp):.4f}")
print(f"recall     :   {baseline_metrics['recall']:.4f}     {float(box.mr):.4f}")

## 9. Evaluate on the TEST split — and why *accuracy* is the wrong metric

We now evaluate on the held-out **test** split (images never seen in training or
early-stopping). For this task we deliberately **do not report accuracy**.

**The accuracy paradox / class imbalance.** Almost every region of a blade image
is healthy surface; real defects are rare. A model that predicts "no defect"
everywhere can score well above 95% accuracy while catching *zero* defects.
Accuracy is dominated by the easy majority (background) and completely hides
failure on the rare class we actually care about.

**Recall is the safety metric.** A missed crack (a false negative) can lead to a
blade failure — costly and dangerous. **Recall** answers the question that
matters: *of all the real defects of this type, how many did we catch?* In
safety-critical inspection we want recall high **per class**, because missing
even a rare defect type (e.g. a lightning strike) is unacceptable. We report
recall per class and explicitly flag the **weakest class**.

**Precision still matters — the trade-off.** Perfect recall with terrible
precision means crying wolf: every image flagged, human reviewers swamped. So we
report precision and F1 too — but recall is the primary lens here.

**AUPRC, not accuracy or ROC.** Under heavy class imbalance the
**precision-recall curve** and its area (**AUPRC**) are far more informative than
accuracy or even ROC-AUC: they focus on the rare positive class and are not
inflated by the abundant easy negatives. We report AUPRC per class.

This section computes, per defect class: **recall, precision, F1, AUPRC**, plus a
**precision-recall curve**, a **confusion matrix**, and **8-10 example images**
(predictions vs ground truth). Everything is saved to `results/`.

> These cells mirror `src/evaluate.py` (the standalone CLI version) so you can
> reproduce the same numbers and figures outside Colab.

In [ ]:
# --- TEST-split evaluation helpers (mirrors src/evaluate.py) ---
import numpy as np, cv2, json
import matplotlib.pyplot as plt

CLASS_NAMES = [names[i] for i in range(len(names))]
NC = len(CLASS_NAMES)
TEST_IMAGES_DIR = DATASET_DIR / 'images' / 'test'
TEST_LABELS_DIR = DATASET_DIR / 'labels' / 'test'
CONF_OP, MATCH_IOU, NMS_IOU, IMGSZ, MIN_CONF = 0.25, 0.5, 0.6, 640, 0.001
IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')

plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 15, 'axes.titlesize': 19, 'axes.titleweight': 'bold',
    'axes.labelsize': 16, 'xtick.labelsize': 13, 'ytick.labelsize': 13, 'legend.fontsize': 12,
})

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), np.float32)
    A, B = a[:, None, :], b[None, :, :]
    ix1 = np.maximum(A[..., 0], B[..., 0]); iy1 = np.maximum(A[..., 1], B[..., 1])
    ix2 = np.minimum(A[..., 2], B[..., 2]); iy2 = np.minimum(A[..., 3], B[..., 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / np.clip(aa[:, None] + ab[None, :] - inter, 1e-9, None)

def load_gt(p, w, h):
    if not p.exists():
        return np.zeros((0, 5), np.float32)
    rows = []
    for line in p.read_text().splitlines():
        s = line.split()
        if len(s) < 5:
            continue
        c, xc, yc, bw, bh = (float(v) for v in s[:5])
        rows.append([c, (xc - bw / 2) * w, (yc - bh / 2) * h, (xc + bw / 2) * w, (yc + bh / 2) * h])
    return np.array(rows, np.float32) if rows else np.zeros((0, 5), np.float32)

class IR:
    __slots__ = ('path', 'w', 'h', 'gt', 'pb', 'pc', 'pcf')
    def __init__(s, path, w, h, gt, pb, pc, pcf):
        s.path, s.w, s.h, s.gt, s.pb, s.pc, s.pcf = path, w, h, gt, pb, pc, pcf

def gather(model):
    imgs = sorted([p for e in IMAGE_EXTS for p in TEST_IMAGES_DIR.rglob('*' + e)], key=lambda p: p.stem)
    out = []
    for ip in imgs:
        r = model.predict(source=str(ip), imgsz=IMGSZ, conf=MIN_CONF, iou=NMS_IOU, verbose=False)[0]
        h, w = r.orig_shape; bx = r.boxes
        if bx is not None and len(bx):
            pb = bx.xyxy.cpu().numpy().astype(np.float32)
            pc = bx.cls.cpu().numpy().astype(int)
            pcf = bx.conf.cpu().numpy().astype(np.float32)
        else:
            pb = np.zeros((0, 4), np.float32); pc = np.zeros((0,), int); pcf = np.zeros((0,), np.float32)
        out.append(IR(ip, w, h, load_gt(TEST_LABELS_DIR / (ip.stem + '.txt'), w, h), pb, pc, pcf))
    return out

def match_scores(res):
    scores = {c: [] for c in range(NC)}; tot = np.zeros(NC, int)
    for r in res:
        for c in range(NC):
            gt = r.gt[r.gt[:, 0] == c][:, 1:5] if len(r.gt) else np.zeros((0, 4), np.float32)
            tot[c] += len(gt); sel = r.pc == c; pb, pcf = r.pb[sel], r.pcf[sel]
            if len(pb) == 0:
                continue
            o = np.argsort(-pcf); pb, pcf = pb[o], pcf[o]
            matched = np.zeros(len(gt), bool); iou = iou_matrix(pb, gt)
            for i in range(len(pb)):
                tp = 0
                if len(gt):
                    j = int(np.argmax(iou[i]))
                    if iou[i, j] >= MATCH_IOU and not matched[j]:
                        matched[j] = True; tp = 1
                scores[c].append((float(pcf[i]), tp))
    return scores, tot

def pr_curve(confs, tps, tot):
    if tot == 0:
        return np.array([0.]), np.array([0.]), float('nan')
    if len(confs) == 0:
        return np.array([0., 0.]), np.array([1., 0.]), 0.0
    o = np.argsort(-confs); tpc = np.cumsum(tps[o]); fpc = np.cumsum(1 - tps[o])
    rec = tpc / tot; prec = tpc / np.clip(tpc + fpc, 1e-9, None)
    env = np.maximum.accumulate(prec[::-1])[::-1]
    rp = np.concatenate([[0.], rec[:-1]])
    return rec, env, float(np.sum((rec - rp) * env))

def op_metrics(scores, tot):
    out = {}
    for c, lst in scores.items():
        if lst:
            a = np.array(lst, np.float32); k = a[:, 0] >= CONF_OP
            tp = int(a[k, 1].sum()); fp = int(k.sum() - tp)
        else:
            tp = fp = 0
        fn = int(tot[c]) - tp
        p = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * p * rec / (p + rec) if p + rec else 0.0
        out[c] = {'precision': p, 'recall': rec, 'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn, 'support': int(tot[c])}
    return out

def confusion(res):
    bg = NC; cm = np.zeros((NC + 1, NC + 1), int)
    for r in res:
        k = r.pcf >= CONF_OP; pb, pc = r.pb[k], r.pc[k]
        gtb = r.gt[:, 1:5] if len(r.gt) else np.zeros((0, 4), np.float32)
        gtc = r.gt[:, 0].astype(int) if len(r.gt) else np.zeros((0,), int)
        iou = iou_matrix(pb, gtb); taken = np.zeros(len(gtb), bool); pm = np.zeros(len(pb), bool)
        order = np.argsort(-r.pcf[k]) if len(pb) else np.array([], int)
        for i in order:
            if len(gtb) == 0:
                continue
            j = int(np.argmax(iou[i]))
            if iou[i, j] >= MATCH_IOU and not taken[j]:
                taken[j] = True; pm[i] = True; cm[gtc[j], pc[i]] += 1
        for j in range(len(gtb)):
            if not taken[j]:
                cm[gtc[j], bg] += 1
        for i in range(len(pb)):
            if not pm[i]:
                cm[bg, pc[i]] += 1
    return cm

print('Helpers ready. TEST images:', TEST_IMAGES_DIR)

In [ ]:
# Run the model on the TEST split and compute recall-first metrics.
test_results = gather(best_model)
scores, total_gt = match_scores(test_results)
op = op_metrics(scores, total_gt)
curves = {}
for c in range(NC):
    a = np.array(scores[c], np.float32) if scores[c] else np.zeros((0, 2), np.float32)
    confs = a[:, 0] if len(a) else np.zeros((0,), np.float32)
    tps = a[:, 1] if len(a) else np.zeros((0,), np.float32)
    curves[c] = pr_curve(confs, tps, int(total_gt[c]))

present = [c for c in range(NC) if total_gt[c] > 0]
per_class = {CLASS_NAMES[c]: {
    'recall': round(op[c]['recall'], 4), 'precision': round(op[c]['precision'], 4),
    'f1': round(op[c]['f1'], 4),
    'auprc': (round(curves[c][2], 4) if not np.isnan(curves[c][2]) else None),
    'support': op[c]['support'], 'tp': op[c]['tp'], 'fp': op[c]['fp'], 'fn': op[c]['fn'],
} for c in range(NC)}
macro = {
    'recall': round(float(np.mean([op[c]['recall'] for c in present])), 4),
    'precision': round(float(np.mean([op[c]['precision'] for c in present])), 4),
    'f1': round(float(np.mean([op[c]['f1'] for c in present])), 4),
    'mAUPRC': round(float(np.mean([curves[c][2] for c in present])), 4),
}
worst = min(present, key=lambda c: op[c]['recall'])

# Recall-sorted table (accuracy intentionally absent).
print(f"TEST per-class metrics @ conf={CONF_OP}  (recall-sorted)")
print(f"{'class':<16}{'recall':>9}{'prec':>8}{'F1':>8}{'AUPRC':>8}{'support':>9}")
for name, m in sorted(per_class.items(), key=lambda kv: kv[1]['recall']):
    ap = f"{m['auprc']:.3f}" if m['auprc'] is not None else '  -  '
    print(f"{name:<16}{m['recall']:>9.3f}{m['precision']:>8.3f}{m['f1']:>8.3f}{ap:>8}{m['support']:>9}")
print(f"{'macro':<16}{macro['recall']:>9.3f}{macro['precision']:>8.3f}{macro['f1']:>8.3f}{macro['mAUPRC']:>8.3f}")
print(f"\n[safety] weakest class by recall: {CLASS_NAMES[worst]} = {op[worst]['recall']:.3f}  <-- most often MISSED")

metrics_test = {
    'split': 'test', 'num_images': len(test_results), 'conf_threshold': CONF_OP, 'match_iou': MATCH_IOU,
    'headline': {'macro_recall': macro['recall'], 'mAUPRC': macro['mAUPRC'],
                 'worst_class_recall': {CLASS_NAMES[worst]: round(op[worst]['recall'], 4)}},
    'per_class': per_class, 'macro': macro,
    'note': 'Accuracy omitted (accuracy paradox under class imbalance); recall & AUPRC are the safety metrics.',
}
(RESULTS_DIR / 'metrics.json').write_text(json.dumps(metrics_test, indent=2))
print('\nSaved ->', RESULTS_DIR / 'metrics.json')

In [ ]:
# Precision-recall curves with per-class AUPRC (presentation-ready).
fig, ax = plt.subplots(figsize=(9, 8)); cmap = plt.get_cmap('tab10')
for c, name in enumerate(CLASS_NAMES):
    rec, prec, ap = curves[c]
    if np.isnan(ap):
        continue
    col = cmap(c % 10)
    ax.plot(rec, prec, lw=2.5, color=col, label=f"{name}  (AUPRC={ap:.3f}, R={op[c]['recall']:.2f})")
    ax.scatter([op[c]['recall']], [op[c]['precision']], color=col, s=70, edgecolor='black', zorder=5)
ax.set_xlabel('Recall  (fraction of real defects found)'); ax.set_ylabel('Precision')
ax.set_title('Per-class Precision-Recall (TEST split)')
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02); ax.grid(True, alpha=0.3)
ax.legend(loc='lower left', framealpha=0.9, title='dot = operating point')
fig.savefig(RESULTS_DIR / 'pr_curve.png', bbox_inches='tight', dpi=150); plt.show()
print('Saved ->', RESULTS_DIR / 'pr_curve.png')

In [ ]:
# Confusion matrix: colour = row-normalised (recall down the diagonal); counts annotated.
cm = confusion(test_results)
labels = CLASS_NAMES + ['background']
norm = cm / np.clip(cm.sum(1, keepdims=True), 1, None)
fig, ax = plt.subplots(figsize=(9.5, 8.5))
im = ax.imshow(norm, cmap='Blues', vmin=0, vmax=1)
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cb.set_label('row-normalised (recall)', rotation=270, labelpad=20)
ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right'); ax.set_yticklabels(labels)
ax.set_xlabel('Predicted'); ax.set_ylabel('Ground truth')
ax.set_title('Detection confusion matrix (TEST split)')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if norm[i, j] > 0.5 else 'black', fontsize=12)
fig.savefig(RESULTS_DIR / 'confusion_matrix_test.png', bbox_inches='tight', dpi=150); plt.show()
print('Saved ->', RESULTS_DIR / 'confusion_matrix_test.png')

In [ ]:
# 8-10 example images: ground truth (left) vs predictions (right), saved to results/examples/.
EX_DIR = RESULTS_DIR / 'examples'; EX_DIR.mkdir(parents=True, exist_ok=True)
cmap = plt.get_cmap('tab10')
chosen = [r for r in test_results if len(r.gt)][:10]

def draw(ax, img, boxes, classes, confs, title):
    ax.imshow(img); ax.set_title(title); ax.axis('off')
    for k in range(len(boxes)):
        x1, y1, x2, y2 = boxes[k]; c = int(classes[k]); col = cmap(c % 10)
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=col, linewidth=2.2))
        lab = CLASS_NAMES[c] if confs is None else f"{CLASS_NAMES[c]} {confs[k]:.2f}"
        ax.text(x1, max(y1 - 4, 0), lab, color='white', fontsize=10,
                bbox=dict(facecolor=col, edgecolor='none', pad=1, alpha=0.85))

for idx, r in enumerate(chosen, 1):
    img = cv2.cvtColor(cv2.imread(str(r.path)), cv2.COLOR_BGR2RGB); k = r.pcf >= CONF_OP
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    draw(axes[0], img, r.gt[:, 1:5], r.gt[:, 0], None, 'Ground truth')
    draw(axes[1], img, r.pb[k], r.pc[k], r.pcf[k], f'Predicted (conf >= {CONF_OP})')
    fig.suptitle(r.path.name, fontsize=14)
    fig.savefig(EX_DIR / f'example_{idx:02d}.png', bbox_inches='tight', dpi=130)
    (plt.show() if idx <= 2 else plt.close(fig))  # show first 2 inline, save the rest
print(f'Saved {len(chosen)} example images ->', EX_DIR)

## Done

Artifacts saved to your Drive under `wtbd_yolo/results/` (and `weights/`):

- `weights/yolov8s_wtbd_best.pt` — the fine-tuned detector.
- `baseline_metrics.json` — pre-training floor (val).
- `per_class_metrics.csv` / `.json` — post-training per-class scores (val).
- `metrics.json` — **TEST**-split recall / precision / F1 / AUPRC per class.
- `pr_curve.png`, `confusion_matrix_test.png` — TEST plots.
- `examples/example_*.png` — ground-truth vs predicted boxes.
- `results.csv`, `results.png` — training curves.

**Next:** feed healthy blade crops into the autoencoder anomaly detector (stage 2 of the project).